# Migration Secrets Setup

Use this notebook after you add your **Service Principal (SP) client ID and client secret** so migration jobs can authenticate to the target workspace.

**How to run it yourself:**
1. **In your terminal** (once): create scope and add your SP client ID and secret (you paste when prompted).
2. **In this notebook**: run the cells below to verify secrets and test the connection.

**What this notebook does:**
- Checks that secret scope `migration_secrets` exists
- Verifies `sp_client_id` and `sp_client_secret` are stored
- Tests connection to the target workspace using SP OAuth

**Keep this notebook PRIVATE — it touches credential setup.**

---

## 1. Configuration

Set your **target** workspace URL and CLI profile below. Use the same profile you use for `databricks bundle deploy -t azure-test --profile <profile>`.

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES (any user: set for your environment)
# ============================================================================

# Your TARGET workspace URL (where dashboards will be migrated TO)
TARGET_WORKSPACE_URL = "https://YOUR-TARGET-WORKSPACE.azuredatabricks.net"  # UPDATE: Your target workspace URL

# Secret scope name (must match databricks.yml sp_secret_scope)
SECRET_SCOPE = "migration_secrets"

# CLI profile for "create scope" / "put-secret" commands (run from your machine)
# Use the profile that points to your SOURCE workspace (e.g. source-azure)
CLI_PROFILE = "source-azure"

# Path to the folder that contains "helpers" (so this notebook can import helpers.sp_oauth_auth).
# Examples: "/Workspace/Users/you@company.com/Migrationtools" or "/Workspace/Repos/you/repo-name"
# Leave empty ("") to auto-detect from this notebook's location (works when notebook is under .../setup-guides/).
BUNDLE_ROOT = ""

# ============================================================================

print("Configuration:")
print(f"  Target Workspace: {TARGET_WORKSPACE_URL}")
print(f"  Secret Scope: {SECRET_SCOPE}")
print(f"  CLI Profile: {CLI_PROFILE}")
print(f"  Bundle root: {BUNDLE_ROOT or '(auto-detect)'}")
print()
if "YOUR-TARGET" in TARGET_WORKSPACE_URL or "cloud.databricks.com" in TARGET_WORKSPACE_URL:
    print("WARNING: Set TARGET_WORKSPACE_URL to your actual target workspace URL if needed.")

## 2. Add your Client ID and Secret (run once in your terminal)

Secrets cannot be written from the notebook. Run these commands **on your machine** (with Databricks CLI configured). You will be prompted to paste each value.

```bash
# Create the scope (if it doesn't exist)
databricks secrets create-scope migration_secrets --profile source-azure

# Paste your Service Principal Application (Client) ID when prompted
databricks secrets put-secret migration_secrets sp_client_id --profile source-azure

# Paste your Service Principal OAuth secret when prompted
databricks secrets put-secret migration_secrets sp_client_secret --profile source-azure
```

Then run the cells below in this notebook to verify and test.

## 3. Check secret scope exists

In [ ]:
# Check if secret scope exists
print("="*80)
print("CHECKING SECRET SCOPE")
print("="*80)

print("\nNote: Secret scopes must be created via CLI (not via dbutils)\n")

try:
    # Try to list the scope (will fail if doesn't exist)
    secrets = dbutils.secrets.list(SECRET_SCOPE)
    print(f"Secret scope '{SECRET_SCOPE}' already exists")
    print(f"   Secrets stored: {len(secrets)}")
except Exception as e:
    print(f"Secret scope '{SECRET_SCOPE}' does NOT exist\n")
    print("To create it, run this CLI command from your terminal:\n")
    print(f"   databricks secrets create-scope {SECRET_SCOPE} --profile {CLI_PROFILE}\n")
    print("Then come back and re-run this cell.")
    raise

print("\n" + "="*80)

If the scope check failed, run the **create-scope** command from **Step 2** in your terminal (use your `CLI_PROFILE`), then re-run the check cell.

## 4. Verify SP credentials (sp_client_id, sp_client_secret)

In [ ]:
# Ensure SP OAuth secrets exist (required for auth_method: sp_oauth in databricks.yml)
required = {"sp_client_id", "sp_client_secret"}
secrets = dbutils.secrets.list(SECRET_SCOPE)
keys = {s.key for s in secrets}
missing = required - keys
if missing:
    print("Missing secrets:", list(missing))
    print("\nRun these in your terminal and paste when prompted:")
    print(f"  databricks secrets put-secret {SECRET_SCOPE} sp_client_id --profile {CLI_PROFILE}")
    print(f"  databricks secrets put-secret {SECRET_SCOPE} sp_client_secret --profile {CLI_PROFILE}")
    raise SystemError("Add sp_client_id and sp_client_secret (see Step 2), then re-run this cell.")
print("SP credentials OK: sp_client_id, sp_client_secret present.")

## 5. List all secrets in scope

In [ ]:
# List stored secrets (values are hidden for security)
print("="*80)
print("SECRETS VERIFICATION")
print("="*80)

secrets = dbutils.secrets.list(SECRET_SCOPE)

print(f"\nSecrets stored in '{SECRET_SCOPE}' scope:\n")
for secret in secrets:
    print(f"   - {secret.key}")

print(f"\nTotal secrets: {len(secrets)}")
print("\n" + "="*80)

## 5b. Add bundle root to path (for importing helpers)

Run this cell once before the connection test. Uses `BUNDLE_ROOT` from config if set; otherwise auto-detects from this notebook's location.

In [ ]:
# Ensure "helpers" can be imported: add bundle root to sys.path (config-driven or auto-detect)
import sys
import os

if BUNDLE_ROOT and BUNDLE_ROOT.strip():
    bundle_root = BUNDLE_ROOT.strip()
    print(f"Using BUNDLE_ROOT from config: {bundle_root}")
else:
    # Auto-detect: this notebook is in .../setup-guides/, so parent of parent = bundle root
    try:
        _nb_path = dbutils.notebook.entry_point.get()
    except Exception:
        _nb_path = os.path.abspath(".")
    bundle_root = os.path.dirname(os.path.dirname(_nb_path))
    print(f"Auto-detected bundle root: {bundle_root}")

if bundle_root not in sys.path:
    sys.path.insert(0, bundle_root)
# Optional check (workspace paths may not exist on local fs)
try:
    _h = os.path.join(bundle_root, "helpers")
    if os.path.exists(_h):
        print("Path set; next cell can import helpers.")
    else:
        print("Path set. If the next cell fails with 'No module named helpers', set BUNDLE_ROOT in the config cell to the folder that contains 'helpers'.")
except Exception:
    print("Path set. Run the next cell to test the connection.")

## 6. Test connection to target workspace (SP OAuth)

In [ ]:
# Test connection to target workspace using SP OAuth (sp_client_id / sp_client_secret)
from helpers.sp_oauth_auth import test_sp_connection

print("="*80)
print("TESTING TARGET WORKSPACE CONNECTION (SP OAuth)")
print("="*80)
print(f"\nTarget: {TARGET_WORKSPACE_URL}\n")

result = test_sp_connection(target_url=TARGET_WORKSPACE_URL, secret_scope=SECRET_SCOPE)

if result["success"]:
    print("CONNECTION TEST PASSED")
    print(f"   Workspace: {result.get('workspace_host', '')}")
    print(f"   Identity: {result.get('user_info', '')}")
    print("\nReady to run migration. Next commands (from your machine):")
    print("   databricks bundle deploy -t azure-test --profile source-azure")
    print("   databricks bundle run generate_deploy -t azure-test --profile source-azure")
else:
    print("CONNECTION FAILED:", result.get("error", ""))
    print("Troubleshooting: see setup-guides/SP_OAUTH_SETUP.md")
    raise SystemError(result.get("error", "SP connection failed"))
print("="*80)

## Optional: PAT token (alternative to SP OAuth)

If you use `auth_method: "pat"` in databricks.yml, store a target workspace PAT via CLI (not from notebook):

```bash
databricks secrets put-secret migration_secrets target_workspace_token --profile source-azure
```
Paste the PAT when prompted. See SP_OAUTH_SETUP.md for SP setup (recommended).

In [ ]:
# No code needed for PAT — use CLI (see cell above). For SP OAuth, use Step 2 and run this notebook.
print("Secrets are managed via CLI. Use Step 2 for SP; optional PAT command in the cell above.")

---

## Setup complete

**Steps you did:** (1) Created scope and added `sp_client_id` / `sp_client_secret` via CLI. (2) Ran this notebook to verify and test.

**Next:** Deploy and run from your machine so code syncs to your workspace folder:

```bash
cd "Customer-Work/Catalog Migration"
databricks bundle deploy -t azure-test --profile source-azure
databricks bundle run generate_deploy -t azure-test --profile source-azure
```

Keep this notebook private. Rotate SP OAuth secret periodically (see SP_OAUTH_SETUP.md).